# 1. Context

> **TODO — slot not started yet.**
>
> Add here: project objective, business question(s) this EDA supports, and what decisions the pipeline design will depend on.

# 2. Data Sources

In [ ]:
import pandas as pd
import os

raw_path = "../data/raw"

os.listdir(raw_path)

In [ ]:
#Loading the taxi trips .csv
trips = pd.read_csv("../data/raw/taxi_trip_data.csv", nrows=100_000)

#Loading the geographics zones .csv
zones = pd.read_csv("../data/raw/taxi_zone_geo.csv")

#Loading "clean"  data
cleaned = pd.read_csv("../data/raw/original_cleaned_nyc_taxi_data_2018.csv", nrows=100_000)

# 3. Overview of the 3 datasets

## Quick Inspection

In [ ]:
def quick(df):
    print("shape:", df.shape)
    print("\ncolumns:")
    print(df.columns)
    print("\ndtypes:")
    print(df.dtypes)
    print("\nisnull:")
    print(df.isnull().sum())
    display(df.head())

## Trips

In [ ]:
quick(trips)

## Zones

In [ ]:
quick(zones)

## Cleaned

In [ ]:
quick(cleaned)

## General view

In [ ]:
datasets = {
    "trips": trips,
    "zones": zones,
    "cleaned": cleaned
}

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nTypes:")
    print(df.dtypes)

We are choosing the "taxi_trip_data" as our main source of information because is the .csv that only has information about the taxi trips, and that is what we are going to analyse in this project

# 4. Relationship between datasets

> **TODO — slot not started yet.**
>
> Explore how `taxi_trip_data` relates to `taxi_zone_geo` (via `pickup_location_id` / `dropoff_location_id`) and how `cleaned` compares to the raw `taxi_trip_data` (same source, pre-cleaned reference?).

# 5. EDA - taxi_trip_data.csv

## 5.1 Structure and types

In [ ]:
import pandas as pd

#Loading the taxi trips .csv
trips = pd.read_csv("../data/raw/taxi_trip_data.csv", nrows=100_000)

In [ ]:
trips.shape

In [ ]:
trips.head()

In [ ]:
trips.info()

"pickup_datetime" and "dropoff_datetime" as object is "wrong" i must change the type for 'date' in the pipeline

payment_type as a number, i found in the description of the dataset the equivalent numbers, so in the future i will need to create a auxiliary table

rate_code is categoric besides beeing int

store_and_fwd_flg is a object but is categoric

In [ ]:
schema = pd.DataFrame({
    "column": trips.columns,
    "dtype": trips.dtypes.values,
    "null_count": trips.isnull().sum().values,
    "unique_count": trips.nunique().values
})

schema

17 columns
100,000 records in the sample
0 nulls across all columns in the sample
8 float64
6 int64
3 object

## 5.2 Data quality

In [ ]:
trips.isnull().sum()

In [ ]:
trips.duplicated().sum()

In [ ]:
trips['payment_type'].value_counts()

In [ ]:
trips["vendor_id"].value_counts()

In [ ]:
trips["rate_code"].value_counts()

In [ ]:
trips[trips["trip_distance"] <= 0].count()

In [ ]:
trips[trips ['pickup_location_id'] == trips['dropoff_location_id']].count()

Doing this data quality analyse we can see some errors:
1) Found 134 lines duplicates
2) Found 1899 lines with trip_distance <= 0
3) Found 4647 lines with pickup_location_id == dropoff_location_id

Now we will analyse this lines to understand whats happening

In [ ]:
trips_wrongs = trips[trips["trip_distance"] <= 0]
trips_wrongs.head()

In [ ]:
trips_wrongs.shape

In [ ]:
duplicates = trips[trips.duplicated(keep=False)]
#keep = false, show from the first apparence to the final, if we keep=True, will only show from the second ocorrence

In [ ]:
duplicates.head()

In [ ]:
duplicates.shape

In [ ]:
duplicates.sort_values(
    by=["pickup_datetime", "dropoff_datetime"]
).head(10)

I identified 134 duplicated lines, we can see that all columns are duplicates so probably are really duplicates but to have more confidence i will run another lines to analyse especifically other columns

Running another checkup i can confirm that is really duplicates, because we can see that ALL columns are duplicates too

In the 100.000 we found 134 duplicates that translate to 268 lines

So in the pipeline this duplicates will be removed

In [ ]:
duplicate_rows = len(duplicates)
percentage = duplicate_rows / len(trips) * 100

print(f"Rows involved in duplication: {duplicate_rows}")
print(f"Percentage: {percentage:.3f}%")

In [ ]:
duplicate_counts = trips.value_counts()

duplicate_counts[duplicate_counts > 1].head(20)

In [ ]:
trips_wrongs = trips[trips["trip_distance"] <= 0]
trips_wrongs.head()

In [ ]:
trips_wrongs.shape

In [ ]:
trips_wrongs_rows = len(trips_wrongs)
percentage = trips_wrongs_rows / len(trips) * 100

print(f"Rows involved in trip_distance <= 0: {trips_wrongs_rows}")
print(f"Percentage: {percentage:.3f}%")

1899 in 100.000, we talking about less than 2% so we can delete this amount with no deal

In [ ]:
trips_location_wrong = trips[trips['pickup_location_id'] == trips["dropoff_location_id"]]

trips_location_wrong.shape

In [ ]:
trips_location_wrong[trips_location_wrong['trip_distance'] <= 0].shape

In [ ]:
trips_location_wrong[trips_location_wrong['trip_distance'] < 0].shape

In [ ]:
trips_location_wrong.head()

In [ ]:
trips_no_distance = trips_location_wrong[trips_location_wrong['trip_distance'] == 0]
trips_no_distance.shape

In [ ]:
trips_no_distance[trips_no_distance["fare_amount"] == 0].shape

In [ ]:
trips_no_distance[trips_no_distance["fare_amount"] < 0].shape

In [ ]:
fare_invalid = trips_no_distance[
    trips_no_distance["fare_amount"] <= 0
]

fare_invalid[[
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "payment_type",
    "rate_code"
]]

## 5.3 Time series analysis

In [ ]:
trips_temp = trips.copy()

trips_temp["pickup_datetime"] = pd.to_datetime(
    trips_temp["pickup_datetime"]
)

trips_temp["dropoff_datetime"] = pd.to_datetime(
    trips_temp["dropoff_datetime"]
)

trips_temp[["pickup_datetime", "dropoff_datetime"]].dtypes

In [ ]:
trips_temp["pickup_datetime"].min()

In [ ]:
trips_temp["pickup_datetime"].max()

In [ ]:
trips_temp["dropoff_datetime"].min()

In [ ]:
trips_temp["dropoff_datetime"].max()

Now we know that we are working in the gap between 01/01/2009 and 01/01/2019 ( in the first 100.000 lines)

In [ ]:
trips_temp["trip_duration"] = (
    trips_temp["dropoff_datetime"]
    - trips_temp["pickup_datetime"]
)

trips_temp["trip_duration_minutes"] = (
    trips_temp["trip_duration"].dt.total_seconds() / 60
)

trips_temp["trip_duration_minutes"].describe()

We can see some errors because how a trip has -30 minutes? So now we will explore more in this

In [ ]:
(trips_temp["trip_duration_minutes"] < 0).sum()

We have 2 trips that have a duration less than 0 minutes

In [ ]:
(trips_temp["trip_duration_minutes"] == 0).sum()

We have 92 trips that have a duration equal 0 minutes

First lets see what is in the negative time data lines

In [ ]:
negative_duration = trips_temp[trips_temp["trip_duration_minutes"] < 0]

negative_duration[
    [
        "pickup_datetime",
        "dropoff_datetime",
        "trip_duration_minutes",
        "trip_distance",
        "fare_amount",
        "pickup_location_id",
        "dropoff_location_id"
    ]
]

Identified 2 lines with negative time duration in 100.000 lines, showing pickup_datetime > dropoff_datetime. Because of the scale of the "problem" beeing so small, we decided to remove this lines in the future pipeline

In [ ]:
zero_duration = trips_temp[
    trips_temp["trip_duration_minutes"] == 0
]

zero_duration[
    [
        "pickup_datetime",
        "dropoff_datetime",
        "trip_duration_minutes",
        "trip_distance",
        "fare_amount",
        "pickup_location_id",
        "dropoff_location_id"
    ]
]

## 5.4 Distributions

> **TODO — slot not started yet.**
>
> Histograms / boxplots for `trip_distance`, `fare_amount`, `trip_duration_minutes`, `tip_amount`, `total_amount`. Check skewness and whether a log scale is needed.

## 5.5 Outliers

> **Partially covered above.** Outliers already surfaced during 5.2 (Data quality) and 5.3 (Time series analysis):
>  - `trip_distance <= 0` → 1,899 rows (~1.9%)
>  - `pickup_location_id == dropoff_location_id` → 4,647 rows
>  - `trip_duration_minutes < 0` → 2 rows
>  - `trip_duration_minutes == 0` → 92 rows
>
> **TODO — remaining work for this slot:**
>
> - Consolidate the outlier findings above into one summary table
> - IQR / z-score based outlier detection for `fare_amount`, `trip_distance`, `trip_duration_minutes` (the checks above were rule-based, not statistical)
> - Boxplots for the numeric columns to visualize the outliers

## 5.6 Relationships between variables

> **TODO — slot not started yet.**
>
> Correlation matrix for numeric columns; scatter plots such as `trip_distance` × `fare_amount`, `trip_duration_minutes` × `trip_distance`; check if `payment_type` relates to `tip_amount`.

# 6. EDA - taxi_zone_geo

## 6.1 Structure

> **TODO — slot not started yet.**
>
> `shape`, `head`, `info`, dtypes for `zones` (beyond the initial `quick(zones)` check in section 3).

## 6.2 Quality

> **TODO — slot not started yet.**
>
> Nulls, duplicates, and value ranges for the zone geography fields.

## 6.3 Zone integrity

> **TODO — slot not started yet.**
>
> Check `location_id` is unique / a valid primary key, no orphan or duplicated zone ids.

# 7. Relationship: Trips × Zones

## 7.1 Pickup zones

> **TODO — slot not started yet.**
>
> Distribution of trips by `pickup_location_id`, top/bottom zones by volume.

## 7.2 Dropoff zones

> **TODO — slot not started yet.**
>
> Same analysis as 7.1, but for `dropoff_location_id`.

## 7.3 Unmatched records

> **TODO — slot not started yet.**
>
> Check whether every `pickup_location_id` / `dropoff_location_id` in `trips` exists in `zones` (anti-join); quantify unmatched rows.

# 8. Conclusions

> **TODO — slot not started yet.**
>
> Summarize the key findings from sections 5–7 into a short narrative once they are complete.

# 9. Pipeline requirements

> **Draft — based on findings so far, to be finalized after sections 5.4–8 are complete.**
>
> - Convert `pickup_datetime` and `dropoff_datetime` from object to datetime
> - Create an auxiliary lookup table for `payment_type` codes
> - Treat `rate_code` and `store_and_fwd_flag` as categorical
> - Remove exact duplicate rows (134 duplicates / 268 rows in the 100k sample)
> - Filter out rows with `trip_distance <= 0` (~1.9% of the 100k sample)
> - Remove rows with negative `trip_duration_minutes` (pickup_datetime > dropoff_datetime)
> - Decide how to handle rows with `trip_duration_minutes == 0` (92 rows) — pending investigation
> - Decide how to handle `pickup_location_id == dropoff_location_id` rows (4,647 rows) — pending investigation